## Funnel Analysis

The goal is to perform funnel analysis for an e-commerce website.

Typically, websites have a clear path to conversion: for instance, you land on the home page, then you search, select a product and buy it. At each of these steps, some users will drop off and leave the site. The sequence of pages that leads to conversion is called ‘funnel’ .

Data Science can have a tremendous impact on funnel optimization.
Funnel analysis allows to understand where/when our users abandon the website. It gives crucial insights on user behavior and on ways to improve the user experience as well as it often allows to discover bugs.

In [0]:
# how to install new modules
#pip install seaborn

![image.png](attachment:image.png)

In [0]:
#importing libraries 

#data analysis modules
import pandas as pd 
import numpy as np

#plotting
import matplotlib.pyplot as plt 
import seaborn as sns

#interacting with operating system
import os



In [0]:
CATALOG_NAME = "workspace"
SCHEMA_NAME = "default"
VOLUME_NAME = "course_data"

# 文件路径应该位于 Volume 中 'BDA2_Data' 文件夹下
# VOLUME_DATA_FOLDER = "BDA2_Data/" 

# 1. 构造完整的 Volume 路径

user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
path = (
    f"/Workspace/Users/{user}/bda_course/BDA2/data/"
)
# 2. 假设你要读取 BDA1/data 文件夹中的某个文件，例如 'some_data_file.csv'
# 假设我们要读取的是 'ab_data.csv' (或者你需要替换成 BDA1 实际的文件名)
# file_name = "ab_data.csv" 
# path = os.path.join(volume_base_path, file_name)#loading the data as dataframe. 
# # path=os.environ['USERPROFILE']+r'\OneDrive\BDA2\Jupyter Notebook\Funnel_Analysis\data/'
# path=os.environ['USERPROFILE']+r'\Documents\BDA2\Jupyter Notebook\Funnel_Analysis\data/'

user_page = pd.read_csv(path+'user_table.csv')
home_page = pd.read_csv(path+'home_page_table.csv')
search_page = pd.read_csv(path+'search_page_table.csv')
payment_page = pd.read_csv(path+'payment_page_table.csv')
confirmation_page = pd.read_csv(path+'payment_confirmation_table.csv')

In [0]:
#having a brief look at the loaded dataframes. 
print(user_page.head())
print(home_page.head())
print(search_page.head())
print(payment_page.head())
print(confirmation_page.head())

You are looking at data from an e-commerce website. The site is very simple and has just 4 pages:


- The first page is the home page. When you come to the site for the first time, you can only land on the home page as a first page.

- From the home page, the user can perform a search and land on the search page. 

- From the search page, if the user clicks on a product, she will get to the payment page, where she is asked to provide payment information in order to buy that product.

- If she does decide to buy, she ends up on the confirmation page


In [0]:
#merging the tables together into one dataframe. 
df = pd.merge(left = user_page , right = home_page, how = 'left' ,on = 'user_id')
df = pd.merge(left = df, right = search_page, how = 'left', on = 'user_id', suffixes = ('_home', '_search'))
df = pd.merge(left = df, right = payment_page, how = 'left', on = 'user_id')
df = pd.merge(left = df, right = confirmation_page, how = 'left', on = 'user_id',suffixes = ('_payment', '_confirmation'))
df.head()

In [0]:
df['page_home'] = df['page_home'].replace({'home_page': 1})
df['page_search'] = df['page_search'].replace({'search_page': 1})
df['page_payment'] = df['page_payment'].replace({'payment_page': 1})
df['page_confirmation'] = df['page_confirmation'].replace({'payment_confirmation_page': 1})

# Replacing NaN values with 0
df = df.fillna(0)

display(df)

In [0]:
total_list = [['page_home',df['page_home'].sum()],['page_search',df['page_search'].sum()],['page_payment',df['page_payment'].sum()],['page_confirmation',df['page_confirmation'].sum()]]

In [0]:
total = pd.DataFrame(total_list, columns = ['page','sum'])
display(total)

In [0]:
total


In [0]:
import matplotlib.pyplot as plt

# Sample data
sizes = total['sum']
labels =total['page']

# Create a pie chart
plt.pie(sizes, labels=labels, autopct='%1.1f%%')

# Add a title
plt.title('Pie Chart')

# Display the chart
plt.show()

In [0]:
#Visualizing barplot for page visits.
# sns.barplot(x ='page', y = 'sum', data = total)

# Create a bar plot with Seaborn
ax = sns.barplot(x='page', y='sum', data=total)

# Add data labels to the bars
for p in ax.patches:
    ax.annotate(format(p.get_height(), '.0f'), (p.get_x() + p.get_width() / 2, p.get_height()), ha = 'center', va = 'center', xytext = (0, 5), textcoords = 'offset points')

# Display the plot
plt.show()

In [0]:
#The function below shows the basic statistical makeup of a specific feature
def stat(df):
    
    '''
    INPUT: Desired Dataframe
    OUTPUT: Dataframe and plot that contains the mean of the conversion of the Input
    '''
    pages = df[['page_home','page_search','page_payment','page_confirmation']]
    mean = []
    for i in pages: 
        mean.append([i,df[i].fillna(0).mean()])
    mean = pd.DataFrame(mean,columns =['page','mean'])

    print(mean)
    
    #plotting the graph of funnel analysis based on the feature
    fig, ax = plt.subplots(figsize=(8, 5))
    ax=sns.barplot(x = 'page', y = 'mean', data = mean)
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.2%}', (p.get_x() + p.get_width() / 2, p.get_height()), ha = 'center', va = 'center', xytext = (0, 5), textcoords = 'offset points')
    ax.set_xlabel('Page', fontsize=12)
    ax.set_ylabel('Ratio of Visitors', fontsize=12)
    plt.show()
    

In [0]:
#Viewing descriptive statistics of 'df' DataFrame
#function allows me to reuse the code 
stat(df)

In [0]:
#Finding the unique values in each feature.
for i in df[['device','sex']]:
    print (df[i].unique())

In [0]:
#Seperating the original dataset into seperate features.
df_desktop = df[df['device']== 'Desktop']
df_mobile = df[df['device'] == 'Mobile']
df_male = df[df['sex'] == 'Male']
df_female = df[df['sex'] == 'Female']

In [0]:
#Visualizing descriptive statisitical make up of seperate features.
print(stat(df_male))
print(stat(df_female))
print(stat(df_desktop))
print(stat(df_mobile))

The above funnel analysis shows that between the homepage and searchpage, the customers churn by 50%. This means that out of all the people that make it to the homepage only half of them will go through the searchpage. 

However, it is more surprising to note that the churn between search page and payment page is greater than half. This is where most of the customers who do make it to the searchpage, ultimately choose to not go through with the payment. This can be due to a variety of reasons. The most important cause being that the search algorithm that is implemented on the website is not effective enough. This is implied from the fact that the greater magnitude of churn indicates that the customer has not found the product that he or she desires. 

There are two main features that were explored inorder to further determine the cause of churn - sex and device. While looking at the difference between the male and female features, it turns out that difference between the churn is not that much, implying that the approach taken to target these two segments of customers is effective, since we are able to convert them, enough though it is not efficient enough. 

However, when we look at the churn of customers depending on their device, it is a different story. While one may assume that the customer churn will be lower on desktop than on mobile simply because, usage of desktop implies the customer is more serious about the purchase rather than the impulsive customer on the mobile. The data above does not support this claim, infact that customer on the mobile is more likely to purchase the product than on desktop. This implies that the payment interface on the mobile is more customer friendly than that of the desktop. 

The above analysis is not the end but rather the starting point of improving the conversion rate of the ecommerce website.